In [5]:
import os
from dotenv import load_dotenv
from datasets import load_dataset

In [6]:
from langchain_astradb import AstraDBVectorStore

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [25]:
embeddings = HuggingFaceEmbeddings(
  model_name='sentence-transformers/all-MiniLM-L6-v2',
  model_kwargs={'device': 'cpu'},
  encode_kwargs={"batch_size": 8}
)

vstore = AstraDBVectorStore(
  collection_name="test",
  embedding=embeddings,
  token=os.getenv("ASTRA_DB_APPLICATION_TOKEN"),
  api_endpoint=os.getenv("ASTRA_DB_API_ENDPOINT")
)
print('astra db vector store configured successfully!')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1447.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


astra db vector store configured successfully!


In [26]:
load_dotenv()

dataset = load_dataset("datastax/philosopher-quotes")["train"]
len(dataset)

450

In [27]:
docs = []
for entry in dataset:
  metadata = {
    "author": entry["author"]
  }

  if entry["tags"]:
    for tag in entry["tags"].split(";"):
      metadata[tag] = "yes"
    
  doc = Document(page_content=entry["quote"], metadata=metadata)
  docs.append(doc)

In [28]:
len(docs)

450

In [29]:
inserted_ids = vstore.add_documents(docs)
print(f"Inserted {len(inserted_ids)} documents.")

Inserted 450 documents.


In [34]:
retriever = vstore.as_retriever(search_kwargs = {"k": 3})
rag_prompt = ChatPromptTemplate.from_template(
  """
    Answer the questions only through the provided context.
    Don't hallucinate.
    Give accurate answers only.
    <context> {context} </context>
    Question: {input}
  """
)

llm = ChatGroq(
  model_name="llama-3.3-70b-versatile"
)


In [41]:
chain = (
  {"context": retriever, "input": RunnablePassthrough()}
  | rag_prompt
  | llm
  | StrOutputParser()
)

In [56]:
response = chain.invoke("In the given context, what is the most important to allow the brain and also provide me the tags?")

In [57]:
response

'In the given context, the most important thing to allow the brain is "the full measure of sleep which is required to restore it." \n\nThe tags associated with this information are: \'author\': \'schopenhauer\', \'ethics\': \'yes\', \'knowledge\': \'yes\'.'

In [58]:
vstore.delete_collection()